<a href="https://colab.research.google.com/github/cpdong/public/blob/master/test/LLMBind_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🧬 LLMBind

**De novo protein binder design from a target structure.**

Given a single-chain target (PDB/CIF), an LLM generates candidate binder sequences, which are
then filtered by a PPI classifier (*bindscan*) and, optionally, a structure filter (*NetSurfP-3.0*).

**Runtime:** this is a PyTorch/CUDA pipeline — set **Runtime → Change runtime type → GPU (T4)**.
_(No TPU / JAX / TensorFlow is used.)_

**Two steps:**
1. **Install** (~4 min, run once)
2. **Generate binders**

> ⚠️ You must supply a **generation model** in step 2 (`llm_model`) — a fine-tuned protein
> language model, given as a local path or a HuggingFace repo id. The demo files do **not**
> include one.


In [ ]:
#@title 1. Install LLMBind  &  fetch demo files (~4 min){ display-mode: "form" }
#@markdown Run this **once**. Installs the PyTorch-based pipeline
#@markdown (`transformers` + `fair-esm` + NetSurfP-3.0) and preloads the ESM weights.
#@markdown No TPU / JAX / TensorFlow needed — uses Colab's built-in CUDA PyTorch.

import os, time, subprocess

t0 = time.time()
WORK_DIR = "/content/LLMBind_demo"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

def sh(cmd):
    print(">>", cmd)
    if subprocess.run(cmd, shell=True).returncode != 0:
        raise RuntimeError("command failed: " + cmd)

if not os.path.isfile(os.path.join(WORK_DIR, "READY")):
    # Python deps. torch / torchvision ship with Colab (CUDA build) — don't reinstall them.
    sh("pip -q install "
       "transformers==4.57.3 accelerate fair-esm "
       "'numpy==1.26.4' scipy==1.13.1 scikit-learn==1.6.1 "
       "pandas h5py 'pyyaml==6.0.2' 'requests==2.32.3' biopython matplotlib")

    # NetSurfP-3.0 (structure filter) — a torch-based fork, no TensorFlow
    sh("pip -q install git+https://github.com/cpdong/NetSurfP_3.0_standalone.git")

    # Demo + model files
    BASE = "https://raw.githubusercontent.com/cpdong/public/refs/heads/master/test"
    for f in ["run_test.py", "bs_model.pt", "PDL1.fasta", "PDL1.pdb"]:
        sh(f"wget -q {BASE}/{f} -O {f}")

    # Preload ESM weights into the torch hub cache
    sh('python -c "import esm; esm.pretrained.esm1b_t33_650M_UR50S()"')  # NetSurfP-3.0
    sh('python -c "import esm; esm.pretrained.esm2_t6_8M_UR50D()"')      # bindscan PPI

    open(os.path.join(WORK_DIR, "READY"), "w").close()
    print("\n✅ LLMBind setup complete.")
else:
    print("Already installed (delete /content/LLMBind_demo/READY to reinstall).")

# numpy was pinned to <2; if it was downgraded after import you may need to
# Runtime → Restart session once, then continue at step 2 (no need to reinstall).
print(f"\nElapsed: {time.time() - t0:.0f}s")


In [ ]:
#@title 2. Generate binders{ display-mode: "form" }

#@markdown ### Target structure
#@markdown Single-chain PDB/CIF. Keep **use_demo_target** on to try the bundled PD-L1 example,
#@markdown or turn on **upload_target** to upload your own.
use_demo_target = True   #@param {type:"boolean"}
upload_target   = False  #@param {type:"boolean"}
target_pdb_path = "/content/target.pdb"  #@param {type:"string"}

#@markdown ### Generation model  (REQUIRED)
#@markdown Local path **or** HuggingFace repo id of your fine-tuned binder-generation LM.
#@markdown The demo does not ship one — set this yourself.
llm_model = ""  #@param {type:"string"}

#@markdown ### Design options
num_designs         = 100  #@param {type:"integer"}
generate_batch_size = 16   #@param {type:"integer"}
min_length          = 50   #@param {type:"integer"}
max_length          = 130  #@param {type:"integer"}

#@markdown ### PPI (bindscan) filter
ppi_threshold = 0.3  #@param {type:"number"}

#@markdown ### Structure filter (NetSurfP-3.0) — optional
#@markdown Turn on and give an `nsp3.pth` weights file to enable it. ESM-1b weights are
#@markdown already cached from step 1. Leave off to skip structure filtering.
enable_structure_filter = False  #@param {type:"boolean"}
nsp3_model              = ""     #@param {type:"string"}
min_structured_fraction = 0.6    #@param {type:"number"}

output_dir = "/content/llmbind_output"  #@param {type:"string"}

# ---------------------------------------------------------------------------
import os, shlex, subprocess
from pathlib import Path

WORK_DIR = "/content/LLMBind_demo"

# Resolve the target structure
if upload_target:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No PDB file uploaded.")
    target_pdb_path = "/content/" + list(uploaded.keys())[0]
elif use_demo_target:
    target_pdb_path = os.path.join(WORK_DIR, "PDL1.pdb")

if not os.path.isfile(target_pdb_path):
    raise FileNotFoundError(f"Target structure not found: {target_pdb_path}")

if not llm_model:
    raise ValueError(
        "`llm_model` is empty. Point it at your fine-tuned generation model "
        "(local path or HuggingFace repo id) — the demo does not include one."
    )

Path(output_dir).mkdir(parents=True, exist_ok=True)

cmd = [
    "python", os.path.join(WORK_DIR, "run_test.py"),
    "--target_pdb",          target_pdb_path,
    "--gen_model",           llm_model,
    "--ppi_model",           os.path.join(WORK_DIR, "bs_model.pt"),
    "--ppi_threshold",       str(ppi_threshold),
    "--num_designs",         str(num_designs),
    "--generate_batch_size", str(generate_batch_size),
    "--min_length",          str(min_length),
    "--max_length",          str(max_length),
    "--output_dir",          output_dir,
]

if enable_structure_filter and nsp3_model:
    cmd += ["--nsp3_model", nsp3_model,
            "--min_structured_fraction", str(min_structured_fraction)]
else:
    cmd += ["--no_structure_filter"]

print("Running:\n  " + " ".join(shlex.quote(x) for x in cmd) + "\n")
subprocess.run(cmd, check=True)

print("\nFinished. Output files:")
for p in sorted(Path(output_dir).rglob("*")):
    if p.is_file():
        print("  ", p)
